Esperimento Embedding Statici (Custom vs Pre-trained)

Questo notebook implementa il confronto tra Word2Vec (addestrato da zero sulle issue) e GloVe (pre-addestrato su Wikipedia) per la classificazione binaria di Issue GitHub (Bug vs Feature).

In [6]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
# NLP & Embedding
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from sklearn.svm import SVC
import gensim.downloader as api

# Machine Learning
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

print("Librerie importate con successo!")

Librerie importate con successo!


Caricamento Dati e Label Encoding

In [2]:
# Caricamento dei dataset (Modifica i percorsi se necessario)
train_df = pd.read_csv('../data/train_set.csv')
test_df = pd.read_csv('../data/test_set.csv')

# Conversione della variabile target ("bug", "feature") in numerica (0, 1)
le = LabelEncoder()
y_train = le.fit_transform(train_df['label'])
y_test = le.transform(test_df['label'])

print(f"Classi mappate: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Dimensioni Train: {train_df.shape} | Dimensioni Test: {test_df.shape}")

Classi mappate: {'bug': 0, 'feature': 1}
Dimensioni Train: (32650, 2) | Dimensioni Test: (13994, 2)


Tokenizzazione Leggera con Rimozione Stopwords

In [3]:
def tokenize_and_filter(text):
    # simple_preprocess converte in minuscolo, toglie punteggiatura e tokenizza
    tokens = simple_preprocess(str(text))
    # Rimuoviamo le stopwords per evitare che "inquinino" la media matematica dei vettori
    return [token for token in tokens if token not in STOPWORDS]

# Applicazione della tokenizzazione
print("Tokenizzazione in corso...")
train_df['tokens'] = train_df['text'].apply(tokenize_and_filter)
test_df['tokens'] = test_df['text'].apply(tokenize_and_filter)
print("Tokenizzazione completata!")

Tokenizzazione in corso...
Tokenizzazione completata!


Word2Vec custom (Addestrato da zero)

In [4]:
from gensim.models import Word2Vec

print("Addestramento di Word2Vec Custom sui dati di Train...")
# Addestriamo SOLO sul train set per evitare data leakage
w2v_model = Word2Vec(
    sentences=train_df['tokens'],
    vector_size=100,
    window=5,
    sg=1,          # Skip-gram
    min_count=2,   # Ignora parole uniche (potenziali refusi)
    workers=4      # Ottimizzato per CPU multi-core
)
print("Addestramento completato!")
print(f"Vocabolario Word2Vec Custom: {len(w2v_model.wv)} parole uniche.")

Addestramento di Word2Vec Custom sui dati di Train...
Addestramento completato!
Vocabolario Word2Vec Custom: 77441 parole uniche.


Funzione di Aggregazione (Sentence Vector) con Gestione OOV

In [5]:
def get_mean_vector(model, tokens, vector_size=100):
    # Estrae i vettori solo per le parole presenti nel vocabolario del modello
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    
    # Se nessuna parola è nel vocabolario (o testo vuoto), restituisce un vettore di zeri
    if not valid_vectors:
        return np.zeros(vector_size)
    
    # Fa la media matematica dei vettori validi
    return np.mean(valid_vectors, axis=0)

# Generazione delle matrici delle feature per Word2Vec Custom
X_train_w2v = np.vstack(train_df['tokens'].apply(lambda x: get_mean_vector(w2v_model, x)))
X_test_w2v = np.vstack(test_df['tokens'].apply(lambda x: get_mean_vector(w2v_model, x)))

print(f"Matrice Train Word2Vec: {X_train_w2v.shape}")
print(f"Matrice Test Word2Vec: {X_test_w2v.shape}")

Matrice Train Word2Vec: (32650, 100)
Matrice Test Word2Vec: (13994, 100)


GloVe Pre-addestrato (Wikipedia 100d)

In [7]:
print("Caricamento di GloVe (glove-wiki-gigaword-100) in corso...")
# Questo modello è leggerissimo (~120MB) ed efficiente per macchine con 8GB di RAM
glove_model = api.load("glove-wiki-gigaword-100")
print("GloVe caricato con successo!")

Caricamento di GloVe (glove-wiki-gigaword-100) in corso...
GloVe caricato con successo!


Generazione dell'Embedding GloVe per le Issue

In [8]:
def get_mean_vector_glove(model, tokens, vector_size=100):
    # KeyedVectors di GloVe usa la sintassi 'word in model' anziché 'word in model.wv'
    valid_vectors = [model[word] for word in tokens if word in model]
    
    if not valid_vectors:
        return np.zeros(vector_size)
    
    return np.mean(valid_vectors, axis=0)

# Generazione delle matrici delle feature per GloVe
X_train_glove = np.vstack(train_df['tokens'].apply(lambda x: get_mean_vector_glove(glove_model, x)))
X_test_glove = np.vstack(test_df['tokens'].apply(lambda x: get_mean_vector_glove(glove_model, x)))

print(f"Matrice Train GloVe: {X_train_glove.shape}")
print(f"Matrice Test GloVe: {X_test_glove.shape}")

Matrice Train GloVe: (32650, 100)
Matrice Test GloVe: (13994, 100)


Addestramento, Tuning e Valutazione dei Modelli
Creiamo una pipeline di addestramento e valutazione riutilizzabile per i 4 esperimenti.

In [9]:
final_results = {}

def run_experiment(exp_name, X_train, y_train, X_test, y_test):
    print(f"\n=== ESECUZIONE ESPERIMENTO: {exp_name} ===")
    
    # --- 1. LOGISTIC REGRESSION ---
    lr = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
    lr_params = {'C': [0.1, 1, 10]}
    grid_lr = GridSearchCV(lr, lr_params, cv=3, scoring='f1_macro', n_jobs=-1)
    grid_lr.fit(X_train, y_train)
    best_lr = grid_lr.best_estimator_
    
    y_pred_lr = best_lr.predict(X_test)
    
    # --- 2. DECISION TREE ---
    dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)
    dt_params = {'max_depth': [5, 10]}
    grid_dt = GridSearchCV(dt, dt_params, cv=3, scoring='f1_macro', n_jobs=-1)
    grid_dt.fit(X_train, y_train)
    best_dt = grid_dt.best_estimator_
    
    y_pred_dt = best_dt.predict(X_test)

    # --- 3. SVM (Support Vector Machine) ---
    svm = SVC(class_weight='balanced', random_state=42)
    # Tuning leggero: kernel lineare e RBF con due valori di C
    svm_params = {'C': [0.1, 1], 'kernel': ['linear', 'rbf']}
    grid_svm = GridSearchCV(svm, svm_params, cv=3, scoring='f1_macro', n_jobs=-1)
    grid_svm.fit(X_train, y_train)
    y_pred_svm = grid_svm.bestestimator.predict(X_test)

    # --- 4. SALVATAGGIO NEL DIZIONARIO GLOBALE ---
    final_results[exp_name] = {}
    
    # Helper function interna per non ripetere il codice delle metriche
    def get_metrics(y_true, y_pred, grid):
        return {
            "best_params": grid.best_params_,
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "precision": float(precision_score(y_true, y_pred, average='macro')),
            "recall": float(recall_score(y_true, y_pred, average='macro')),
            "f1_score": float(f1_score(y_true, y_pred, average='macro')),
            "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
        }

    final_results[exp_name]["logistic_regression"] = get_metrics(y_test, y_pred_lr, grid_lr)
    final_results[exp_name]["decision_tree"] = get_metrics(y_test, y_pred_dt, grid_dt)
    final_results[exp_name]["svm"] = get_metrics(y_test, y_pred_svm, grid_svm)
    
    # --- 5. PLOTTING  ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5)) # Aumentata larghezza
    class_labels = le.classes_
    
    models = ["logistic_regression", "decision_tree", "svm"]
    titles = ["Logistic Regression", "Decision Tree", "SVM"]
    cmaps = ["Blues", "Greens", "Oranges"]

    for i, model_key in enumerate(models):
        sns.heatmap(final_results[exp_name][model_key]["confusion_matrix"], 
                    annot=True, fmt='d', cmap=cmaps[i], ax=axes[i],
                    xticklabels=class_labels, yticklabels=class_labels)
        axes[i].set_title(f'{exp_name}\n{titles[i]}')
        axes[i].set_ylabel('Classe Reale')
        axes[i].set_xlabel('Classe Predetta')
    
    plt.tight_layout()
    plt.savefig(f'../figures/confusion_matrix_{exp_name.lower()}.png')
    plt.show()

    print(f"Esperimento {exp_name} completato.")

Esecuzione dei 6 Esperimenti

In [10]:
#Esegui Word2Vec
run_experiment("Word2Vec", X_train_w2v, y_train, X_test_w2v, y_test)

#Esegui GloVe
run_experiment("GloVe", X_train_glove, y_train, X_test_glove, y_test)


=== ESECUZIONE ESPERIMENTO: Word2Vec ===


KeyboardInterrupt: 

Salvataggio risultati

In [ ]:
# Salvataggio Word2Vec
output_path_w2v = "../results/word2vec_results.json"
with open(output_path_w2v, 'w', encoding='utf-8') as f:
    json.dump(final_results["Word2Vec"], f, indent=4, ensure_ascii=False)

# Salvataggio GloVe
output_path_glove = "../results/glove_results.json"
with open(output_path_glove, 'w', encoding='utf-8') as f:
    json.dump(final_results["GloVe"], f, indent=4, ensure_ascii=False)

print("File JSON generati correttamente dai risultati globali.")


Risultati Word2Vec salvati in: ../results/word2vec_results.json
Risultati GloVe salvati in: ../results/glove_results.json
